# Fieldwise — Agricultural Advisory Control Room

This notebook introduces the project, walks through its multi-agent architecture, inspects the bundled mandi-price snapshot, and demonstrates the same market-provider function used by the Market Agent.

> **Requirements:** Run the notebook with the Python environment in `backend/.venv`. The live market example reads `backend/.env`; it never prints API credentials.


## 1. Project overview

Fieldwise is a full-stack agricultural advisory control room. A farmer submits crop, location, growth stage, and farm details. Specialist agents gather weather, crop, market, scheme, and risk findings. A coordinator combines the findings into a plain-language advisory and records the sources and limitations.

The application uses React and Vite in the client, FastAPI in the backend, MongoDB for accounts and advisory state, ChromaDB for scheme-document retrieval, and the Microsoft Agent Framework for orchestration. Provider credentials belong in the backend environment only.

## 2. How an advisory is coordinated

The workflow starts with intake and then fans out to specialist work. Each specialist publishes a finding to a shared persistent blackboard; the coordinator reads those findings and builds the final response. Weather and market data are provider facts, while the LLM synthesizes explanations from supplied findings.

| Component | Responsibility |
|---|---|
| Intake Agent | Validates the farmer input and initializes the session |
| Weather Agent | Adds local weather context when a provider is configured |
| Crop Agent | Recommends stage-aware field actions |
| Market Agent | Uses live AGMARKNET data when available, with the bundled CSV as fallback |
| Scheme Agent | Retrieves relevant passages from indexed government documents |
| Risk Alert Agent | Assesses risk based on verified findings and flags missing evidence |
| Advisor Coordinator | Combines findings into one advisory, preserving sources and limitations |

Agent progress and findings are persisted so the interface can show what completed or failed.

## 3. Set up the notebook environment

Select the `backend/.venv` Python interpreter as the notebook kernel. From the project root, install the backend package if needed with `backend/.venv/bin/pip install -e backend`. The following cell makes the backend modules importable and loads settings from `backend/.env` without displaying secrets.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'backend' / 'app').is_dir():
    PROJECT_ROOT = Path.cwd().resolve().parent
BACKEND_DIR = PROJECT_ROOT / 'backend'
if not (BACKEND_DIR / 'app').is_dir():
    raise FileNotFoundError('Open this notebook from the project root or its parent directory.')
sys.path.insert(0, str(BACKEND_DIR))
os.chdir(BACKEND_DIR)  # Settings loads backend/.env relative to this directory.
print('Project:', PROJECT_ROOT.name)
print('Backend modules:', (BACKEND_DIR / 'app').is_dir())

## 4. Explore the bundled market snapshot

The CSV is a dated fallback dataset. The adapter selects the latest matching crop records, prefers a location match where possible, and does not invent a historical trend when only one date is available.

In [ ]:
import csv
from collections import Counter

market_csv = BACKEND_DIR / 'app' / 'data' / 'AgriAI.csv'
with market_csv.open(newline='', encoding='utf-8-sig') as f:
    csv_rows = list(csv.DictReader(f))
print(f'{len(csv_rows):,} rows in {market_csv.name}')
print('Columns:', ', '.join(csv_rows[0].keys()) if csv_rows else 'No rows')
print('Most common commodities:', Counter(r['Commodity'] for r in csv_rows).most_common(8))
for row in csv_rows[:3]:
    print({k: row[k] for k in ('State', 'District', 'Market', 'Commodity', 'Arrival_Date', 'Modal_x0020_Price')})

## 5. Demonstrate the Market Agent provider path

This calls `fetch_market`, the provider function invoked by the backend's Market Agent. It makes a generic Rice query using the configured data.gov.in resource and key. If the live request cannot be used, the adapter can return the CSV fallback and marks that with `live_api_fallback`. This cell reports only source and record metadata; it never prints the key.

A successful API response confirms the provider returned usable rows to the adapter. The full LLM agent and advisory coordinator are exercised when you create an advisory in the application.

In [ ]:
import asyncio
from app.core.config import get_settings
from app.services.providers import fetch_market

settings = get_settings()
print('Market URL configured:', bool(settings.market_api_url))
print('Market key configured:', bool(settings.market_api_key))

market_result = await fetch_market(
    settings,
    {'crop': 'Rice', 'location': 'India', 'crop_stage': 'Sowing'},
)
for field in ('available', 'status', 'source', 'live_api_fallback', 'record_count', 'price_date', 'location_scope'):
    print(f'{field}: {market_result.get(field)}')
if not market_result.get('available'):
    print('Reason:', market_result.get('reason'))

### How to read the result

- `source: data.gov.in AGMARKNET live API` means the backend adapter selected live API records.
- `live_api_fallback: True` means the request did not yield usable live records and the bundled CSV supplied the result.
- `location_scope: national` means the adapter used a transparent India-wide fallback when it found no location match.
- Market prices are reported in INR per quintal. A single dated snapshot is not enough to claim a price trend.

## 6. Backend API and knowledge retrieval

The React client communicates with FastAPI over authenticated routes. Useful endpoints include `/api/health`, `/api/advisory/create`, advisory session/status/blackboard routes, and admin knowledge-source routes. Check `docs/architecture.md` and the interactive `/docs` page for the current API surface.

Administrators can add PDF or TXT scheme references. The backend extracts and chunks text, indexes it in ChromaDB, and the Scheme Agent cites retrieved passages. Scanned PDFs need OCR before upload; eligibility and benefits should be checked against current official guidance.

## 7. Run the application

From the project root, install dependencies and configure `backend/.env` using `backend/.env.example`. Start MongoDB, then run:

```sh
npm run dev
```

The frontend runs at `http://localhost:5173`; the backend and its interactive API documentation are at `http://127.0.0.1:8000` and `http://127.0.0.1:8000/docs`. Keep API keys server-side and never include them in notebook output.

## 8. Production and advisory limitations

A production deployment also needs managed MongoDB and ChromaDB, strong secrets, HTTPS, explicit CORS origins, backups, monitoring, and configured weather, market, and LLM providers. Provider availability can vary; inspect each advisory's agent status, source, and limitations. This tool supports farm decisions but does not replace local agricultural extension advice or official scheme eligibility checks.